In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

$$ f=x+1 $$
$$ \frac{\partial f}{\partial x} = 1 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=3+f $$
$$ \frac{\partial F}{\partial f} = 1 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} = 1$$

In [ ]:
class Add1Function(autograd.Function):
    """Function to add 1 to the input tensor."""

    @staticmethod
    def forward(ctx, x):
        return x + 1

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert dF_df.shape == (1,)
        # assert dF_df.item() == 1.0

        df_dx = 1
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   

class Add1(nn.Module):
    """Module to add 1 to the input tensor."""

    def forward(self, x):
        return Add1Function.apply(x)
    
    def backward(self, *_):
        """
        Autograd never calls `backward` on modules — 
        only `autograd.Function` objects participate in the computational graph.
        """
        
        assert(False)
   

def _test_Add1():
    x = tr.tensor([4.0], dtype=tr.float32, requires_grad=True)
    F = 3 + Add1()(x)     
    F.backward()     

    assert F.item() == 3.0 + (4.0 + 1)
    assert x.grad.item() == 1.0


if __name__ == "__main__":
    _test_Add1()

$$ f=2x $$
$$ \frac{\partial f}{\partial x} = 2 $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=1+3f $$
$$ \frac{\partial F}{\partial f} = 3 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$

In [ ]:
class Mul2Function(autograd.Function):
    """Function to multiply the input tensor by 2."""

    @staticmethod
    def forward(ctx, x):
        return 2 * x

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert dF_df.shape == (1,)
        # assert dF_df.item() == 3.0
        
        df_dx = 2
        dF_dx = dF_df * df_dx

        return (dF_dx, )
   

class Mul2(nn.Module):
    """Module to multiply the input tensor by 2."""
    
    def forward(self, x):
        return Mul2Function.apply(x)
   
    def backward(self, *_):
        """
        Autograd never calls `backward` on modules — 
        only `autograd.Function` objects participate in the computational graph.
        """
   
        assert False


def _test_Mul2():
    x = tr.tensor([4.0], requires_grad=True)
    F = 1 + 3 * Mul2()(x)     
    F.backward()     

    assert F.item() == 1.0 + 3.0 * (2.0 * 4.0)
    assert x.grad.item() == 3.0 * 2.0


if __name__ == "__main__":
    _test_Mul2()

$$ f=xy $$
$$ \frac{\partial f}{\partial x} = y $$
$$ \frac{\partial f}{\partial y} = x $$
$$ \diamond \diamond \diamond $$
$$ \text{Let } F=2+5f $$
$$ \frac{\partial F}{\partial f} = 5 $$
$$ \frac{\partial F}{\partial x} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial x} $$
$$ \frac{\partial F}{\partial y} = \frac{\partial F}{\partial f} \cdot \frac{\partial f}{\partial y} $$

In [ ]:
class MultiplyFunction(autograd.Function):
    """Function to multiply two input tensors."""

    @staticmethod
    def forward(ctx, x, y):
        ctx.save_for_backward(x, y)
        return x * y

    @staticmethod
    def backward(ctx, dF_df):
        # For this example only
        # assert dF_df.shape == (1,)
        # assert dF_df.item() == 5.0

        (x, y) = ctx.saved_tensors

        df_dx = y
        dF_dx =  dF_df * df_dx

        df_dy = x
        dF_dy =  dF_df * df_dy

        return (dF_dx, dF_dy)
   

class Multiply(nn.Module):
    """Module to multiply two input tensors."""

    def forward(self, x, y):
        return MultiplyFunction.apply(x, y)
   
    def backward(self, *_):
        """
        Autograd never calls `backward` on modules — 
        only `autograd.Function` objects participate in the computational graph.
        """
        
        assert False


def _test_MultiplyFunction():
    x = tr.tensor([3.0], requires_grad=True)
    y = tr.tensor([4.0], requires_grad=True)

    F = 2.0 + 5.0 * Multiply()(x, y)     
    F.backward()        

    assert F.item() == 2.0 + 5.0 * (3.0 * 4.0)
    assert x.grad.item() == 4.0 * 5.0
    assert y.grad.item() == 3.0 * 5.0


if __name__ == "__main__":
    _test_MultiplyFunction()

$$ z=xW $$
$$ p=\frac{e^z}{1+e^z} $$
$$ L=-y\log p - (1-y)\log(1-p) $$
$$ F = E(L) $$

$$ \diamond \diamond \diamond $$

$$ \frac{\partial L}{\partial z} = (p-y) $$

$$ 
\frac{\partial L}{\partial x} =
\frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial x} =
(p-y) \cdot W^T 
$$

$$ 
\frac{\partial L}{\partial W} = 
\frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial W} = 
x^T \cdot(p-y) 
$$

In [ ]:
class ChainRuleFunction(autograd.Function):
    """Example of a function that implements the chain rule for derivatives of a composite function."""

    @staticmethod
    def forward(ctx, x, W, y):
      z = x @ W
      p = tr.exp(z) / (1 + tr.exp(z))
      L = -y * tr.log(p) - (1-y) * tr.log(1-p)

      ctx.save_for_backward(x, W, y, z, p)

      return L

    @staticmethod
    def backward(ctx, grad_output):
      (x, W, y, z, p) = ctx.saved_tensors
      (S, FI, FO) = (x.shape[0], x.shape[1], W.shape[1])

      dL_dz = (p - y)

      # For chain rule we use Hadamard product, as gradients accumulate multiplicatively.
      dL_dz = dL_dz * grad_output

      # `tr.mean` is outside of this function,
      # so its participation to a gradient is provided by `grad_output`.
      assert grad_output == approx(tr.full(grad_output.shape, 1.0 / S / FO, dtype=grad_output.dtype))
     
      # (S, FO) @ (FI, FO).T -> (S, FI)
      dL_dx = dL_dz @ W.t()
      assert dL_dx.shape == x.shape

      # (S, FI).T @ (S, FO) -> (FI, FO)
      dL_dW = x.t() @ dL_dz
      assert dL_dW.shape == W.shape
     
      return (dL_dx, dL_dW, None)


def _test_ChainRuleFunction():
  S = 5   # Samples
  FI = 3  # Features in
  FO = 2  # Features out

  x = tr.randn((S, FI), requires_grad=True, dtype=tr.float32)
  W = tr.randn((FI, FO), requires_grad=True, dtype=tr.float32)
  y = tr.randn((S, FO), dtype=tr.float32)
 
  L = tr.mean(ChainRuleFunction.apply(x, W, y))
  L.backward()


if __name__ == "__main__":
  _test_ChainRuleFunction() 